# Классификация текстов с помощью Gradient Boosting Machines(GBM) и AdaBoost(Адаптивный бустинг)

## 1.Загрузка необходимых данных и библиотек

In [51]:
import re
import pandas as pd
from datasets import load_dataset

from sklearn.datasets import fetch_20newsgroups

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

from sklearn.metrics import f1_score

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1.1 Загрузка датасета emotion

In [52]:
dataset_emotion_train = load_dataset('emotion', split='train')
dataset_emotion_test = load_dataset('emotion', split='test')

print(f"emotion train size: {len(dataset_emotion_train)}")
print(f"emotion test size: {len(dataset_emotion_test)}")

emotion train size: 16000
emotion test size: 2000


### 1.2 Загрузка датачета 20newsgroups(только указанные  категории)

In [53]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
dataset_news_test = fetch_20newsgroups(subset='test', categories=categories,shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news_train.data)}")
print(f"news test size: {len(dataset_news_test.data)}")

news train size: 2345
news test size: 1561


## 2. Предобработка текстов

### 2.1 Удаление шума(для news)

In [54]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text= re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?']", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 2.2 Токенизация, стемминг, лемматизация, лемматизация(только сущ и прил)

In [55]:
def get_wordnet_pos(tag):
    """Преобразует тег POS из NLTK в формат WordNet."""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [56]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

def stem_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

def lemmatize_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmarized_words = []
    for word, tag in pos_tags:
        if word not in stop_words:
            wordnet_tag = get_wordnet_pos(tag)
            lemmarized_words.append(lemmatizer.lemmatize(word, pos=wordnet_tag))
    return ' '.join(lemmarized_words)

def lemmatize_noun_adj_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmarized_words = []
    for word, tag in pos_tags:
        if word not in stop_words:
            if tag.startswith('NN') or tag.startswith('J'):
                wordnet_tag = get_wordnet_pos(tag)
                lemmarized_words.append(lemmatizer.lemmatize(word, pos=wordnet_tag))
    return ' '.join(lemmarized_words)

In [57]:
preprocessors = {
    'tokenization': tokenize_text,
    'stemming': stem_text,
    'lemmatization': lemmatize_text,
    'lemmatization_noun_adj': lemmatize_noun_adj_text
}

### 2.3 Применение предобработки

In [58]:
emotion_train_texts = dataset_emotion_train['text']
emotion_train_labels = dataset_emotion_train['label']
emotion_test_texts = dataset_emotion_test['text']
emotion_test_labels = dataset_emotion_test['label']

In [59]:
news_train_texts = [clean_noise(text) for text in dataset_news_train.data]
news_test_texts = [clean_noise(text) for text in dataset_news_test.data]
news_train_labels = dataset_news_train.target
news_test_labels = dataset_news_test.target

In [60]:
processed_train = {'emotion': {}, 'news': {}}
processed_test = {'emotion': {}, 'news': {}}

In [61]:
for prep_name, prep_func in preprocessors.items():
    print(f"Processing emotion with {prep_name}...")
    processed_train['emotion'][prep_name] = [prep_func(text) for text in emotion_train_texts]
    processed_test['emotion'][prep_name] = [prep_func(text) for text in emotion_test_texts]

Processing emotion with tokenization...
Processing emotion with stemming...
Processing emotion with lemmatization...
Processing emotion with lemmatization_noun_adj...


In [62]:
for prep_name, prep_func in preprocessors.items():
    print(f"Processing emotion with {prep_name}...")
    processed_train['news'][prep_name] = [prep_func(text) for text in news_train_texts]
    processed_test['news'][prep_name] = [prep_func(text) for text in news_test_texts]

Processing emotion with tokenization...
Processing emotion with stemming...
Processing emotion with lemmatization...
Processing emotion with lemmatization_noun_adj...


## 3. Векторизаторы и классификаторы

In [63]:
vectorizers = {
    'binary': CountVectorizer(binary=True),
    'count': CountVectorizer(binary=False),
    'tfidf': TfidfVectorizer()
}

classifiers = {
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'AdaBoostClassifier': AdaBoostClassifier(random_state=42)
}

In [64]:
results = []

## 4. Цикл экспериментов (по всем датасетам, предобработкам, векторизаторам и классификаторам)

In [65]:
for dataset_name in ['emotion', 'news']:
    print(f"\n=============== ДАТАСЕТ:{dataset_name.upper()} ==============")
    y_train = emotion_train_labels if dataset_name == 'emotion' else news_train_labels
    y_test = emotion_test_labels if dataset_name == 'emotion' else news_test_labels

    for prep_name in preprocessors.keys():
        X_train_prep = processed_train[dataset_name][prep_name]
        X_test_prep = processed_test[dataset_name][prep_name]

        for vec_name, vectorizer in vectorizers.items():
            print(f"\nПредобработка: {prep_name}, Векторизация: {vec_name}")
            X_train_vec = vectorizer.fit_transform(X_train_prep)
            X_test_vec = vectorizer.transform(X_test_prep)

            for clf_name, clf in classifiers.items():
                clf.fit(X_train_vec, y_train)
                y_pred = clf.predict(X_test_vec)

                f1_micro = f1_score(y_test, y_pred, average='micro')
                f1_macro = f1_score(y_test, y_pred, average='macro')
                f1_weighted = f1_score(y_test, y_pred, average='weighted')

                results.append({
                    'Dataset': dataset_name,
                    'Preprocessing': prep_name,
                    'Vectorizer': vec_name,
                    'Classifier': clf_name,
                    'F1_micro': f1_micro,
                    'F1_macro': f1_macro,
                    'F1_weighted': f1_weighted
                })
                print(f"{clf_name}: micro={f1_micro:.4f}, macro={f1_macro:.4f}, weighted={f1_weighted:.4f}")


=============== ДАТАСЕТ:EMOTION ==============

Предобработка: tokenization, Векторизация: binary
GradientBoostingClassifier: micro=0.8445, macro=0.8001, weighted=0.8452
AdaBoostClassifier: micro=0.3480, macro=0.0900, weighted=0.1867

Предобработка: tokenization, Векторизация: count
GradientBoostingClassifier: micro=0.8440, macro=0.8002, weighted=0.8445
AdaBoostClassifier: micro=0.3480, macro=0.0900, weighted=0.1867

Предобработка: tokenization, Векторизация: tfidf
GradientBoostingClassifier: micro=0.8440, macro=0.8003, weighted=0.8441
AdaBoostClassifier: micro=0.3480, macro=0.0896, weighted=0.1858

Предобработка: stemming, Векторизация: binary
GradientBoostingClassifier: micro=0.7895, macro=0.7536, weighted=0.7915
AdaBoostClassifier: micro=0.3540, macro=0.1042, weighted=0.1955

Предобработка: stemming, Векторизация: count
GradientBoostingClassifier: micro=0.7900, macro=0.7513, weighted=0.7920
AdaBoostClassifier: micro=0.3540, macro=0.1042, weighted=0.1955

Предобработка: stemming, Ве

## 5. Таблица результатов

In [66]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='F1_micro', ascending=False)
df_results

,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
0,emotion,tokenization,binary,GradientBoostingClassifier,0.844500,0.800090,0.845171
2,emotion,tokenization,count,GradientBoostingClassifier,0.844000,0.800222,0.844545
4,emotion,tokenization,tfidf,GradientBoostingClassifier,0.844000,0.800254,0.844057
12,emotion,lemmatization,binary,GradientBoostingClassifier,0.798500,0.763087,0.799273
14,emotion,lemmatization,count,GradientBoostingClassifier,0.798000,0.763983,0.798897
16,emotion,lemmatization,tfidf,GradientBoostingClassifier,0.798000,0.763347,0.799372
10,emotion,stemming,tfidf,GradientBoostingClassifier,0.797000,0.761556,0.798384
8,emotion,stemming,count,GradientBoostingClassifier,0.790000,0.751277,0.792030
6,emotion,stemming,binary,GradientBoostingClassifier,0.789500,0.753600,0.791545
30,news,stemming,binary,GradientBoostingClassifier,0.763613,0.766636,0.766565


## 6. Выводы

### 1. Датасеты

- emotion имеет лучшие показатели, т.к. в датасете меньше шума
- news, несмотря на очистку, имеет много технической лексики, длинные сообщения

### 2. Предобработка текста

- простая токенизация на emotion показывает наилучшие результаты: на коротких текстах большое влияние даёт полная форма слова
- лемматизация и стемминг показывают на emotion результаты похуже
- лемматизация по сущ и прил на emotion совсем не передаёт смысл
- на news лучшие результаты даёт стемминг и лемматизация: упрощение сложных технических терминов без потери ключевой информации
- лемматизация по сущ и прил на news тоже снижает качество но не так сильно, в технических текстах большую роль играет сущ и прил

### 3. Тип векторизации

- на emotion все три типа векторизации работают примерно одинаково
- на news binary и count показывают себя получше tf-idf, но не сильно

### 4. Классификаторы

- Gradient Boosting значительно превосходит AdaBoost на обоих датасетах(У Ada базовая глубина Decision tree =1, а у GBM =3)

### 5. Лучшие комбинации

- emotion:	tokenization + binary/count/tf-idf + GradientBoostingClassifier
- news:	stemming + binary + GradientBoostingClassifier